# 04 — Modelos: Daniel Velasco
Experimentos con **Logistic Regression** (paramétrico) y **LinearSVC** (SVM) usando la mejor codificación encontrada en el notebook 02.


In [10]:
import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import mlflow
import mlflow.sklearn
from dotenv import load_dotenv
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
)

warnings.filterwarnings('ignore')
load_dotenv()

RANDOM_SEED = int(os.getenv('RANDOM_SEED', 42))
MLFLOW_TRACKING_URI    = os.getenv('MLFLOW_TRACKING_URI', 'http://ec2-52-203-38-164.compute-1.amazonaws.com:5000')
MLFLOW_EXPERIMENT_NAME = os.getenv('MLFLOW_EXPERIMENT_NAME', 'sentiment140')
MEMBER = 'daniel'

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
np.random.seed(RANDOM_SEED)

DATA_ENCODED = Path('../data/encoded')
MODELS_DIR   = Path('../models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print('Setup complete — member:', MEMBER)

Setup complete — member: daniel


## Carga de datos

Usamos TF-IDF bigrama (`tfidf_bi`) como encoding — cambiar a `tfidf_uni` o `bow` si ese fue el mejor en notebook 02.

In [11]:
# Change 'tfidf_bi' to 'tfidf_uni' or 'bow' based on best encoding from notebook 02
ENCODING = 'tfidf_bi'

X_train_full = sp.load_npz(DATA_ENCODED / f'X_train_{ENCODING}.npz')
X_test       = sp.load_npz(DATA_ENCODED / f'X_test_{ENCODING}.npz')
y_train_full = np.load(DATA_ENCODED / 'y_train.npy')
y_test       = np.load(DATA_ENCODED / 'y_test.npy')

# 10 % del train se reserva como validación (estratificado)
VAL_SIZE = 0.10
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=VAL_SIZE, random_state=RANDOM_SEED, stratify=y_train_full
)

print(f'Encoding : {ENCODING}')
print(f'Train    : {X_train.shape}')
print(f'Val      : {X_val.shape}')
print(f'Test     : {X_test.shape}')

Encoding : tfidf_bi
Train    : (1224000, 100000)
Val      : (136000, 100000)
Test     : (240000, 100000)


In [12]:
def _metrics(y_true, y_pred, prefix: str) -> dict:
    """Compute accuracy/precision/recall/f1 and prefix all keys."""
    return {
        f'{prefix}_accuracy':  accuracy_score(y_true, y_pred),
        f'{prefix}_precision': precision_score(y_true, y_pred, average='macro'),
        f'{prefix}_recall':    recall_score(y_true, y_pred, average='macro'),
        f'{prefix}_f1_macro':  f1_score(y_true, y_pred, average='macro'),
    }


def log_run(run_name: str, clf, params: dict, encoding: str) -> dict:
    """Train, evaluate (train / val / test) and log a classifier to MLflow.

    Returns a dict with all metrics, the fitted clf, and test predictions.
    """
    with mlflow.start_run(run_name=run_name) as run:
        mlflow.set_tag('member', MEMBER)
        mlflow.set_tag('stage', 'models')
        mlflow.set_tag('model', type(clf).__name__)
        mlflow.set_tag('encoding', encoding)

        mlflow.log_params({'model': type(clf).__name__, 'encoding': encoding,
                           'random_seed': RANDOM_SEED, **params})

        clf.fit(X_train, y_train)

        train_preds = clf.predict(X_train)
        val_preds   = clf.predict(X_val)
        test_preds  = clf.predict(X_test)

        train_metrics = _metrics(y_train, train_preds, 'train')
        val_metrics   = _metrics(y_val,   val_preds,   'val')
        test_metrics  = _metrics(y_test,  test_preds,  'test')

        # MLflow only stores train + test; val is used locally for model selection
        mlflow.log_metrics({**train_metrics, **test_metrics})
        mlflow.sklearn.log_model(clf, 'model')

        # Log model card artifact
        model_card = {
            'model_name': type(clf).__name__,
            'encoding': encoding,
            'parameters': params,
            'member': MEMBER,
            'stage': 'models',
            'task': 'Binary Sentiment Classification',
            'dataset': 'Sentiment140',
            'metrics': {
                'train': {k.replace('train_', ''): round(v, 4) for k, v in train_metrics.items()},
                'val':   {k.replace('val_',   ''): round(v, 4) for k, v in val_metrics.items()},
                'test':  {k.replace('test_',  ''): round(v, 4) for k, v in test_metrics.items()},
            },
            'created_at': pd.Timestamp.now().isoformat(),
        }
        mlflow.log_dict(model_card, 'model_card.json')

        run_id = run.info.run_id

    result = {**train_metrics, **val_metrics, **test_metrics}
    result['run_id']      = run_id
    result['model']       = type(clf).__name__
    result['run_name']    = run_name
    result['params']      = params
    result['test_preds']  = test_preds
    result['clf']         = clf
    return result

## Modelo 1 — Logistic Regression (grid de hiperparámetros)

Se prueban distintos valores de regularización `C`. Para cada uno se registran métricas de **train**, **validación** y **test** en MLflow y se guarda el modelo en disco.

In [13]:
LR_C_VALUES = [0.01, 0.1, 1.0, 5.0, 10.0]

lr_all_results = []

for c_val in LR_C_VALUES:
    params = {
        'C': c_val,
        'solver': 'saga',
        'max_iter': 1000,
        'n_jobs': -1,
        'random_state': RANDOM_SEED,
    }
    run_name = f'lr_{ENCODING}_C{c_val}'
    clf = LogisticRegression(**params)
    res = log_run(run_name, clf, params, ENCODING)
    lr_all_results.append(res)

    print(f'C={c_val:6}  '
          f'train_f1={res["train_f1_macro"]:.4f}  '
          f'val_f1={res["val_f1_macro"]:.4f}  '
          f'test_f1={res["test_f1_macro"]:.4f}  '
          f'run_id={res["run_id"]}')

# Mejor LR según F1 en validación
lr_best = max(lr_all_results, key=lambda x: x['val_f1_macro'])
print(f'\nMejor LR → C={lr_best["params"]["C"]}  val_f1={lr_best["val_f1_macro"]:.4f}')

2026/03/09 04:40:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/09 04:40:31 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run lr_tfidf_bi_C0.01 at: http://ec2-52-203-38-164.compute-1.amazonaws.com:5000/#/experiments/11/runs/798052a2ef5d44618a391b17ec89b191
🧪 View experiment at: http://ec2-52-203-38-164.compute-1.amazonaws.com:5000/#/experiments/11
C=  0.01  train_f1=0.7798  val_f1=0.7799  test_f1=0.7810  run_id=798052a2ef5d44618a391b17ec89b191


2026/03/09 04:40:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/09 04:40:57 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run lr_tfidf_bi_C0.1 at: http://ec2-52-203-38-164.compute-1.amazonaws.com:5000/#/experiments/11/runs/c99a9dd622aa4bf98b7973d049040bbe
🧪 View experiment at: http://ec2-52-203-38-164.compute-1.amazonaws.com:5000/#/experiments/11
C=   0.1  train_f1=0.8142  val_f1=0.8085  test_f1=0.8095  run_id=c99a9dd622aa4bf98b7973d049040bbe


2026/03/09 04:41:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/09 04:41:23 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run lr_tfidf_bi_C1.0 at: http://ec2-52-203-38-164.compute-1.amazonaws.com:5000/#/experiments/11/runs/cb31dd0e673e484780ec20cd6d515411
🧪 View experiment at: http://ec2-52-203-38-164.compute-1.amazonaws.com:5000/#/experiments/11
C=   1.0  train_f1=0.8404  val_f1=0.8212  test_f1=0.8219  run_id=cb31dd0e673e484780ec20cd6d515411


2026/03/09 04:41:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/09 04:41:58 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run lr_tfidf_bi_C5.0 at: http://ec2-52-203-38-164.compute-1.amazonaws.com:5000/#/experiments/11/runs/48c86cea51594a4286bbfc838a78c831
🧪 View experiment at: http://ec2-52-203-38-164.compute-1.amazonaws.com:5000/#/experiments/11
C=   5.0  train_f1=0.8502  val_f1=0.8191  test_f1=0.8187  run_id=48c86cea51594a4286bbfc838a78c831


2026/03/09 04:42:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/09 04:42:53 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run lr_tfidf_bi_C10.0 at: http://ec2-52-203-38-164.compute-1.amazonaws.com:5000/#/experiments/11/runs/8eeda39e4de840ff9244bb5e8d0d7896
🧪 View experiment at: http://ec2-52-203-38-164.compute-1.amazonaws.com:5000/#/experiments/11
C=  10.0  train_f1=0.8520  val_f1=0.8159  test_f1=0.8164  run_id=8eeda39e4de840ff9244bb5e8d0d7896

Mejor LR → C=1.0  val_f1=0.8212


## Modelo 2 — LinearSVC / SVM (grid de hiperparámetros)

Igual que con LR: se barre `C` y se registran métricas de train / val / test.

In [14]:
SVM_C_VALUES = [0.01, 0.1, 1.0, 5.0, 10.0]

svm_all_results = []

for c_val in SVM_C_VALUES:
    params = {
        'C': c_val,
        'max_iter': 2000,
        'random_state': RANDOM_SEED,
    }
    run_name = f'svm_{ENCODING}_C{c_val}'
    # CalibratedClassifierCV wraps LinearSVC para tener predict_proba
    svm_base = LinearSVC(**params)
    clf = CalibratedClassifierCV(svm_base, cv=3)
    res = log_run(run_name, clf, params, ENCODING)
    svm_all_results.append(res)

    print(f'C={c_val:6}  '
          f'train_f1={res["train_f1_macro"]:.4f}  '
          f'val_f1={res["val_f1_macro"]:.4f}  '
          f'test_f1={res["test_f1_macro"]:.4f}  '
          f'run_id={res["run_id"]}')

# Mejor SVM según F1 en validación
svm_best = max(svm_all_results, key=lambda x: x['val_f1_macro'])
print(f'\nMejor SVM → C={svm_best["params"]["C"]}  val_f1={svm_best["val_f1_macro"]:.4f}')

2026/03/09 04:43:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/09 04:43:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run svm_tfidf_bi_C0.01 at: http://ec2-52-203-38-164.compute-1.amazonaws.com:5000/#/experiments/11/runs/572c93ab1f0e400296caa946ff7f884a
🧪 View experiment at: http://ec2-52-203-38-164.compute-1.amazonaws.com:5000/#/experiments/11
C=  0.01  train_f1=0.8092  val_f1=0.8046  test_f1=0.8061  run_id=572c93ab1f0e400296caa946ff7f884a


2026/03/09 04:44:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/09 04:44:30 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run svm_tfidf_bi_C0.1 at: http://ec2-52-203-38-164.compute-1.amazonaws.com:5000/#/experiments/11/runs/f4e6345008eb4bd0a0cc05138193f3fd
🧪 View experiment at: http://ec2-52-203-38-164.compute-1.amazonaws.com:5000/#/experiments/11
C=   0.1  train_f1=0.8371  val_f1=0.8208  test_f1=0.8212  run_id=f4e6345008eb4bd0a0cc05138193f3fd


2026/03/09 04:48:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/09 04:48:08 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run svm_tfidf_bi_C1.0 at: http://ec2-52-203-38-164.compute-1.amazonaws.com:5000/#/experiments/11/runs/c9d627fc8033448ea244d671d4a5cab5
🧪 View experiment at: http://ec2-52-203-38-164.compute-1.amazonaws.com:5000/#/experiments/11
C=   1.0  train_f1=0.8503  val_f1=0.8173  test_f1=0.8174  run_id=c9d627fc8033448ea244d671d4a5cab5


2026/03/09 04:52:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/09 04:52:20 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run svm_tfidf_bi_C5.0 at: http://ec2-52-203-38-164.compute-1.amazonaws.com:5000/#/experiments/11/runs/294a80928d1e45afa3e7b5a868cc4e8a
🧪 View experiment at: http://ec2-52-203-38-164.compute-1.amazonaws.com:5000/#/experiments/11
C=   5.0  train_f1=0.8516  val_f1=0.8122  test_f1=0.8115  run_id=294a80928d1e45afa3e7b5a868cc4e8a


2026/03/09 04:57:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/09 04:57:34 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run svm_tfidf_bi_C10.0 at: http://ec2-52-203-38-164.compute-1.amazonaws.com:5000/#/experiments/11/runs/aa2bf88e56cf40e89c0640930c7ac5e0
🧪 View experiment at: http://ec2-52-203-38-164.compute-1.amazonaws.com:5000/#/experiments/11
C=  10.0  train_f1=0.8516  val_f1=0.8106  test_f1=0.8101  run_id=aa2bf88e56cf40e89c0640930c7ac5e0

Mejor SVM → C=0.1  val_f1=0.8208
